# Tập 4.5 · Deep RL: luyện robot trong mô phỏng 🧠🤖

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap45/tap45_deeprl.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 4 · ⏱ ~40 phút · Giáo trình: Principles of Robot Autonomy, [Chương 18](https://porabook.com/book/reinforcement-learning/) (Mục 18.5)

Mạng nơ-ron thay bảng Q, luyện bằng REINFORCE trong mô phỏng. Trong notebook này bạn:

1. Đánh giá mạng đã luyện sẵn của video (sạch và "thật hơn").
2. **Tự viết** reward-to-go, hàm mất mát REINFORCE và vòng cập nhật (3 bài); ô chấm luyện mạng bằng code của bạn (~30 giây).
3. Xem robot trước và sau khi học.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap45")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import torch
import deeprl as goc               # môi trường + mạng + REINFORCE của video
from kiem_tra import nap
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
pi = nap("pi_sach_0.pt")            # mạng luyện 500 lần trong video
print("mô phỏng sạch :", goc.danh_gia(pi, 1000, seed=500))
print("thật hơn (nhiễu):", goc.danh_gia(pi, 1000, seed=900, nhieu=True))
print("số tham số:", goc.so_tham_so(pi))

## 2. Bài tập

$R_t = \sum_{t'\ge t}\gamma^{t'-t}r_{t'}$, $\quad A = (R - \bar R)/\sigma_R$, $\quad \text{loss} = -\frac1B\sum_{t,i}\log\pi_\theta(u_t^i \mid x_t^i)\,A_t^i\,m_t^i$

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `reward_to_go(rew, gamma)` | tổng thưởng từ bước t về sau |
| **2** | `loi_reinforce(logp, R, alive)` | hàm mất mát có baseline |
| **3** | `mot_cap_nhat(pi, opt, env, gamma)` | chạy, tính loss, cập nhật mạng |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np
import torch

from deeprl import chay_tap  # noqa: F401  (chạy B tập song song, trong ../sim)


def reward_to_go(rew, gamma):
    """Bài 1: tổng thưởng từ bước t về sau, có chiết khấu:  R_t = Σ_{t' ≥ t} γ^{t'−t}·r_{t'}  (cho từng robot).

    Gợi ý: đi ngược từ bước cuối: acc = rew[t] + gamma·acc; R[t] = acc. Trả về mảng cùng hình rew (float32)."""
    raise NotImplementedError("TODO Bài 1: reward-to-go")


def loi_reinforce(logp, R, alive):
    """Bài 2 (Eq. 18.31–18.32, biến thể "trừ baseline" của bài Cart-pole): hàm mất mát để PyTorch lấy gradient.

    Chỉ tính các ô còn chạy (m = alive):
        b = trung bình R[m];   A = (R − b) / (độ lệch chuẩn R[m] + 1e-8)        (ưu thế: tốt hơn hay tệ hơn mức trung bình)
        loss = − Σ_{t, i} logp[t][i] · A[t, i] · m[t, i]  /  B                 (B = số robot)
    Giảm loss = tăng xác suất của lệnh dẫn tới kết quả tốt hơn trung bình. Trả về tensor số (có gradient)."""
    raise NotImplementedError("TODO Bài 2: hàm mất mát REINFORCE")


def mot_cap_nhat(pi, opt, env, gamma=0.99):
    """Bài 3: một lần cập nhật chính sách.

    1. logp, rew, alive, _ = chay_tap(pi, env)      (có sẵn, đã import ở đầu file)
    2. R = reward_to_go(rew, gamma);  loss = loi_reinforce(logp, R, alive)
    3. opt.zero_grad(); loss.backward(); opt.step()
    Trả về (rew, env.ket) để theo dõi (env.ket: 1 tới đích, 2 va chạm, 0 hết giờ)."""
    raise NotImplementedError("TODO Bài 3: một lần cập nhật")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
ve(None, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Mạng luyện sạch hay luyện có ngẫu nhiên hoá miền chịu nhiễu tốt hơn? Đánh giá cả 6 mạng của video.

In [ ]:
for loai in ["sach", "dr"]:
    for sd in range(3):
        pi = nap(f"pi_{loai}_{sd}.pt")
        s, t = goc.danh_gia(pi, 500, seed=500 + sd), goc.danh_gia(pi, 500, seed=900 + sd, nhieu=True)
        print(f"{loai:4s} seed {sd}: sạch {s['toi'] * 100:5.1f}% · thật hơn {t['toi'] * 100:5.1f}%")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap45).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)